# 01 EDA - BahnDelayStory

Objective: complete Phase 3 by profiling coverage, delay distributions, train-type patterns, station/time effects, cancellations, and route outliers before formal analysis.

Success criteria:
- At least 15 plots, each with a caption.
- Coverage break is visible and handled explicitly.
- Surprises and confusing points are documented for Phase 4.

Important scope note: the local dataset currently contains 2025 only. The original question says "since July 2024"; this notebook profiles 2025 and sets up the coverage-break strategy. Download 2024 files later if the final story must cover July-December 2024.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

from bahn_delay_story.config import PROCESSED_DIR, source_parquet_files  # noqa: E402
from bahn_delay_story.pipeline import duckdb_string_literal  # noqa: E402
from bahn_delay_story.quality import verify_database  # noqa: E402

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

WEEKDAY_LABELS = {
    "0": "Sun",
    "1": "Mon",
    "2": "Tue",
    "3": "Wed",
    "4": "Thu",
    "5": "Fri",
    "6": "Sat",
}
WEEKDAY_ORDER = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
SELECTED_TRAIN_TYPES = ["ICE", "IC", "EC", "RE", "RB", "S"]

TABLE_FILES = {
    "stops": "stops_clean.parquet",
    "station_day": "station_day_metrics.parquet",
    "train_day": "train_type_day_metrics.parquet",
    "hourly": "hourly_delay_metrics.parquet",
    "lines": "line_metrics.parquet",
}

missing_outputs = [name for name in TABLE_FILES.values() if not (PROCESSED_DIR / name).exists()]
if missing_outputs:
    raise FileNotFoundError(f"Missing processed outputs: {missing_outputs}. Run the pipeline first.")

con = duckdb.connect()
for view_name, file_name in TABLE_FILES.items():
    path_sql = duckdb_string_literal(str(PROCESSED_DIR / file_name))
    con.execute(f"CREATE OR REPLACE VIEW {view_name} AS SELECT * FROM read_parquet({path_sql})")


def query(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()


def apply_default_layout(fig: go.Figure, height: int = 440) -> go.Figure:
    fig.update_layout(
        template="plotly_white",
        height=height,
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        legend_title_text="",
    )
    return fig


def as_percent_axis(fig: go.Figure, axis: str = "y") -> go.Figure:
    fig.update_layout(**{f"{axis}axis_tickformat": ".0%"})
    return fig

source_files = source_parquet_files()
print(f"Source files: {len(source_files)}")
if source_files:
    print(f"First source: {source_files[0].name}; last source: {source_files[-1].name}")

quality_report = verify_database()
quality_report["clean"]

## Plan

This EDA is descriptive. The main risk is mistaking coverage changes for real performance changes, so the notebook starts with coverage, then moves to train type, delay distributions, time windows, stations, and routes.

Hypotheses checked here:
- H1: long-distance trains have higher late share and heavier tail delays than local services.
- H2: delay pressure concentrates in specific hub stations and evening/time windows.
- H3: cancellation rates and delay minutes tell different stories.

## Coverage And Data Quality

In [ ]:
coverage_month = query("""
SELECT
  service_month,
  COUNT(*) AS row_count,
  COUNT(DISTINCT station_name) AS station_count,
  COUNT(DISTINCT train_type) AS train_type_count,
  AVG(CASE WHEN is_canceled THEN 1.0 ELSE 0.0 END) AS cancellation_share,
  AVG(CASE WHEN is_late_6_min THEN 1.0 ELSE 0.0 END) AS late_share_6_min,
  AVG(delay_min) AS avg_delay_min,
  QUANTILE_CONT(delay_min, 0.9) AS p90_delay_min
FROM stops
GROUP BY 1
ORDER BY 1
""")
coverage_month

In [ ]:
fig = px.bar(
    coverage_month,
    x="service_month",
    y="row_count",
    text_auto=".2s",
    labels={"service_month": "Month", "row_count": "Stop rows"},
    title="Monthly Source Row Count",
)
apply_default_layout(fig)
fig.show()

**Caption - Figure 1.** Monthly row count shows the coverage break: January-October sit near 2M rows, then November and December jump above 13M because the source expands beyond the original large-station set.

In [ ]:
fig = px.line(
    coverage_month,
    x="service_month",
    y="station_count",
    markers=True,
    labels={"service_month": "Month", "station_count": "Distinct stations"},
    title="Distinct Stations By Month",
)
apply_default_layout(fig)
fig.show()

**Caption - Figure 2.** Station coverage is stable at 107 stations through October and then expands to more than 5,000 stations, so raw national trends after October are not comparable without a stable-station sensitivity check.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=coverage_month["service_month"],
    y=coverage_month["late_share_6_min"],
    mode="lines+markers",
    name="Late share 6+ min",
))
fig.add_trace(go.Scatter(
    x=coverage_month["service_month"],
    y=coverage_month["cancellation_share"],
    mode="lines+markers",
    name="Cancellation share",
))
fig.update_layout(
    title="Monthly Late Share And Cancellation Share",
    xaxis_title="Month",
    yaxis_title="Share of stops",
)
as_percent_axis(apply_default_layout(fig))
fig.show()

**Caption - Figure 3.** Late share rises from winter into October, while cancellation share is lower and follows a different pattern, confirming that delays and cancellations need separate metrics.

In [ ]:
stable_vs_all = query("""
WITH station_month AS (
  SELECT
    station_name,
    EXTRACT(MONTH FROM service_date) AS service_month,
    SUM(stop_count) AS stop_count,
    SUM(stop_count * late_share_6_min) AS late_stops,
    SUM(stop_count * cancellation_share) AS canceled_stops
  FROM station_day
  WHERE station_name IS NOT NULL
  GROUP BY 1, 2
), stable AS (
  SELECT station_name
  FROM station_month
  WHERE service_month BETWEEN 1 AND 10
  GROUP BY 1
  HAVING COUNT(DISTINCT service_month) = 10
), all_month AS (
  SELECT
    service_month,
    'All covered stations' AS cohort,
    SUM(stop_count) AS stop_count,
    SUM(late_stops) / SUM(stop_count) AS late_share_6_min,
    SUM(canceled_stops) / SUM(stop_count) AS cancellation_share
  FROM station_month
  GROUP BY 1
), stable_month AS (
  SELECT
    sm.service_month,
    'Stable Jan-Oct stations' AS cohort,
    SUM(sm.stop_count) AS stop_count,
    SUM(sm.late_stops) / SUM(sm.stop_count) AS late_share_6_min,
    SUM(sm.canceled_stops) / SUM(sm.stop_count) AS cancellation_share
  FROM station_month sm
  INNER JOIN stable USING (station_name)
  GROUP BY 1
)
SELECT * FROM all_month
UNION ALL
SELECT * FROM stable_month
ORDER BY service_month, cohort
""")

fig = px.line(
    stable_vs_all,
    x="service_month",
    y="late_share_6_min",
    color="cohort",
    markers=True,
    labels={"service_month": "Month", "late_share_6_min": "Late share 6+ min"},
    title="All Stations Versus Stable-Station Late Share",
)
as_percent_axis(apply_default_layout(fig))
fig.show()

**Caption - Figure 4.** The stable-station line is the safer trend baseline: after the November coverage expansion, all-station late share shifts partly because thousands of new stations enter the data.

In [ ]:
quality_bars = pd.DataFrame([
    {"metric": "Null station names", "rows": quality_report["clean"]["null_station_names"]},
    {"metric": "Cleaned delay nulls", "rows": quality_report["clean"]["null_delay_min"]},
    {"metric": "Out-of-range delays after cleaning", "rows": quality_report["clean"]["out_of_range_delays"]},
    {"metric": "Canceled but flagged late", "rows": quality_report["clean"]["canceled_but_late"]},
])
fig = px.bar(
    quality_bars,
    x="metric",
    y="rows",
    text_auto=True,
    labels={"metric": "Quality check", "rows": "Rows"},
    title="Cleaning And Quality Check Counts",
)
apply_default_layout(fig)
fig.update_xaxes(tickangle=-20)
fig.show()

**Caption - Figure 5.** Quality issues are small relative to 49M rows: station-name nulls and delay outliers exist, but key fields and late-flag consistency pass the hard assertions.

## Delay Distribution

In [ ]:
delay_hist = query("""
SELECT
  CASE
    WHEN delay_min IS NULL THEN 'null'
    WHEN delay_min < -10 THEN '< -10'
    WHEN delay_min BETWEEN -10 AND -1 THEN '-10..-1'
    WHEN delay_min = 0 THEN '0'
    WHEN delay_min BETWEEN 1 AND 5 THEN '1..5'
    WHEN delay_min BETWEEN 6 AND 15 THEN '6..15'
    WHEN delay_min BETWEEN 16 AND 30 THEN '16..30'
    WHEN delay_min BETWEEN 31 AND 60 THEN '31..60'
    ELSE '60+'
  END AS delay_bucket,
  COUNT(*) AS stop_count
FROM stops
GROUP BY 1
""")
bucket_order = ['< -10', '-10..-1', '0', '1..5', '6..15', '16..30', '31..60', '60+', 'null']
delay_hist["delay_bucket"] = pd.Categorical(delay_hist["delay_bucket"], bucket_order, ordered=True)
delay_hist = delay_hist.sort_values("delay_bucket")

fig = px.bar(
    delay_hist,
    x="delay_bucket",
    y="stop_count",
    text_auto=".2s",
    labels={"delay_bucket": "Delay bucket (minutes)", "stop_count": "Stops"},
    title="Stop-Level Delay Distribution",
)
apply_default_layout(fig)
fig.show()

**Caption - Figure 6.** Most stop events are on time or only a few minutes late; the story is driven by the right tail rather than the center of the distribution.

In [ ]:
delay_quantiles = query("""
SELECT
  train_type,
  COUNT(*) AS stop_count,
  QUANTILE_CONT(delay_min, 0.5) AS p50_delay_min,
  QUANTILE_CONT(delay_min, 0.9) AS p90_delay_min,
  QUANTILE_CONT(delay_min, 0.99) AS p99_delay_min
FROM stops
WHERE delay_min IS NOT NULL
GROUP BY 1
HAVING COUNT(*) >= 100000
ORDER BY p90_delay_min DESC
LIMIT 15
""")
quantile_long = delay_quantiles.melt(
    id_vars=["train_type", "stop_count"],
    value_vars=["p50_delay_min", "p90_delay_min", "p99_delay_min"],
    var_name="quantile",
    value_name="delay_min",
)
fig = px.bar(
    quantile_long,
    x="train_type",
    y="delay_min",
    color="quantile",
    barmode="group",
    labels={"train_type": "Train type", "delay_min": "Delay minutes", "quantile": "Quantile"},
    title="Delay Tail By Train Type",
)
apply_default_layout(fig)
fig.show()

**Caption - Figure 7.** Long-distance and several regional operators have much heavier p90/p99 tails than their medians suggest, so average-only reporting would hide the severe-delay pattern.

## Train-Type Patterns

In [ ]:
train_type_summary = query("""
SELECT
  train_type,
  is_long_distance,
  COUNT(*) AS stop_count,
  AVG(CASE WHEN is_late_6_min THEN 1.0 ELSE 0.0 END) AS late_share_6_min,
  AVG(CASE WHEN is_late_15_min THEN 1.0 ELSE 0.0 END) AS late_share_15_min,
  AVG(CASE WHEN is_late_60_min THEN 1.0 ELSE 0.0 END) AS late_share_60_min,
  AVG(CASE WHEN is_canceled THEN 1.0 ELSE 0.0 END) AS cancellation_share,
  AVG(delay_min) AS avg_delay_min,
  QUANTILE_CONT(delay_min, 0.9) AS p90_delay_min
FROM stops
GROUP BY 1, 2
HAVING COUNT(*) >= 100000
ORDER BY stop_count DESC
""")
train_type_summary.head(20)

In [ ]:
volume_top = train_type_summary.sort_values("stop_count", ascending=False).head(15)
fig = px.bar(
    volume_top,
    x="train_type",
    y="stop_count",
    color="is_long_distance",
    text_auto=".2s",
    labels={"train_type": "Train type", "stop_count": "Stops", "is_long_distance": "Long-distance"},
    title="Largest Train Types By Stop Volume",
)
apply_default_layout(fig)
fig.show()

**Caption - Figure 8.** S-Bahn, RB, and RE dominate stop volume, so overall averages are mostly local-service weighted unless the analysis stratifies by train type.

In [ ]:
late_top = train_type_summary.sort_values("late_share_6_min", ascending=False).head(15)
fig = px.bar(
    late_top,
    x="train_type",
    y="late_share_6_min",
    color="is_long_distance",
    text_auto=".1%",
    labels={"train_type": "Train type", "late_share_6_min": "Late share 6+ min", "is_long_distance": "Long-distance"},
    title="Highest Late Share By Train Type",
)
as_percent_axis(apply_default_layout(fig))
fig.show()

**Caption - Figure 9.** ICE and IC sit near the top of the late-share ranking among high-volume train types, supporting the long-distance-delay hypothesis.

In [ ]:
cancel_top = train_type_summary.sort_values("cancellation_share", ascending=False).head(15)
fig = px.bar(
    cancel_top,
    x="train_type",
    y="cancellation_share",
    color="is_long_distance",
    text_auto=".1%",
    labels={"train_type": "Train type", "cancellation_share": "Cancellation share", "is_long_distance": "Long-distance"},
    title="Highest Cancellation Share By Train Type",
)
as_percent_axis(apply_default_layout(fig))
fig.show()

**Caption - Figure 10.** Cancellation rankings are not the same as late-share rankings, which reinforces treating cancellation as a separate outcome.

In [ ]:
fig = px.scatter(
    train_type_summary,
    x="late_share_6_min",
    y="cancellation_share",
    size="stop_count",
    color="is_long_distance",
    hover_name="train_type",
    labels={
        "late_share_6_min": "Late share 6+ min",
        "cancellation_share": "Cancellation share",
        "stop_count": "Stops",
        "is_long_distance": "Long-distance",
    },
    title="Delay Versus Cancellation By Train Type",
)
as_percent_axis(as_percent_axis(apply_default_layout(fig), "x"))
fig.show()

**Caption - Figure 11.** Train types can be cancellation-heavy, delay-heavy, or both; this scatter prevents collapsing operational reliability into one metric.

In [ ]:
segment_summary = query("""
SELECT
  CASE WHEN is_long_distance THEN 'Long-distance' ELSE 'Other services' END AS segment,
  COUNT(*) AS stop_count,
  AVG(CASE WHEN is_late_6_min THEN 1.0 ELSE 0.0 END) AS late_share_6_min,
  AVG(CASE WHEN is_late_15_min THEN 1.0 ELSE 0.0 END) AS late_share_15_min,
  AVG(CASE WHEN is_canceled THEN 1.0 ELSE 0.0 END) AS cancellation_share,
  AVG(delay_min) AS avg_delay_min,
  QUANTILE_CONT(delay_min, 0.9) AS p90_delay_min
FROM stops
GROUP BY 1
""")
segment_long = segment_summary.melt(
    id_vars=["segment", "stop_count"],
    value_vars=["late_share_6_min", "late_share_15_min", "cancellation_share"],
    var_name="metric",
    value_name="share",
)
fig = px.bar(
    segment_long,
    x="metric",
    y="share",
    color="segment",
    barmode="group",
    text_auto=".1%",
    labels={"metric": "Metric", "share": "Share", "segment": "Service segment"},
    title="Long-Distance Versus Other Services",
)
as_percent_axis(apply_default_layout(fig))
fig.show()

**Caption - Figure 12.** Long-distance services have much higher late shares than other services, while the cancellation gap is smaller; H1 is plausible and should be quantified in Phase 4.

In [ ]:
selected_monthly = query("""
SELECT
  EXTRACT(MONTH FROM service_date) AS service_month,
  train_type,
  SUM(stop_count) AS stop_count,
  SUM(stop_count * late_share_6_min) / SUM(stop_count) AS late_share_6_min,
  SUM(stop_count * cancellation_share) / SUM(stop_count) AS cancellation_share
FROM train_day
WHERE train_type IN ('ICE', 'IC', 'EC', 'RE', 'RB', 'S')
GROUP BY 1, 2
ORDER BY 1, 2
""")
fig = px.line(
    selected_monthly,
    x="service_month",
    y="late_share_6_min",
    color="train_type",
    markers=True,
    labels={"service_month": "Month", "late_share_6_min": "Late share 6+ min", "train_type": "Train type"},
    title="Monthly Late Share For Selected Train Types",
)
as_percent_axis(apply_default_layout(fig))
fig.show()

**Caption - Figure 13.** Selected train types do not move in lockstep; the final story should identify which train families drive changes rather than only showing an aggregate trend.

## Time-Window Patterns

In [ ]:
hour_summary = query("""
SELECT
  hour_of_day,
  SUM(stop_count) AS stop_count,
  SUM(stop_count * late_share_6_min) / SUM(stop_count) AS late_share_6_min,
  SUM(stop_count * cancellation_share) / SUM(stop_count) AS cancellation_share,
  SUM(stop_count * avg_delay_min) / SUM(stop_count) AS avg_delay_min
FROM hourly
GROUP BY 1
ORDER BY 1
""")
fig = px.line(
    hour_summary,
    x="hour_of_day",
    y="late_share_6_min",
    markers=True,
    labels={"hour_of_day": "Hour of day", "late_share_6_min": "Late share 6+ min"},
    title="Late Share By Hour Of Day",
)
as_percent_axis(apply_default_layout(fig))
fig.show()

**Caption - Figure 14.** Late share is highest around the evening peak for all services, matching the hypothesis that time windows matter.

In [ ]:
weekday_hour = query("""
SELECT
  weekday,
  hour_of_day,
  SUM(stop_count) AS stop_count,
  SUM(stop_count * late_share_6_min) / SUM(stop_count) AS late_share_6_min
FROM hourly
GROUP BY 1, 2
""")
weekday_hour["weekday_label"] = weekday_hour["weekday"].map(WEEKDAY_LABELS)
heat = weekday_hour.pivot(index="weekday_label", columns="hour_of_day", values="late_share_6_min").reindex(WEEKDAY_ORDER)
fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    labels={"x": "Hour of day", "y": "Weekday", "color": "Late share 6+ min"},
    title="Weekday-Hour Late Share Heatmap",
)
fig.update_layout(coloraxis_colorbar_tickformat=".0%")
apply_default_layout(fig, height=470)
fig.show()

**Caption - Figure 15.** The weekday-hour heatmap shows repeated late-share pressure in late afternoon and evening windows rather than a single isolated day.

In [ ]:
weekday_hour_long = query("""
SELECT
  weekday,
  hour_of_day,
  SUM(stop_count) AS stop_count,
  SUM(stop_count * late_share_6_min) / SUM(stop_count) AS late_share_6_min
FROM hourly
WHERE is_long_distance
GROUP BY 1, 2
""")
weekday_hour_long["weekday_label"] = weekday_hour_long["weekday"].map(WEEKDAY_LABELS)
heat_long = weekday_hour_long.pivot(index="weekday_label", columns="hour_of_day", values="late_share_6_min").reindex(WEEKDAY_ORDER)
fig = px.imshow(
    heat_long,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    labels={"x": "Hour of day", "y": "Weekday", "color": "Late share 6+ min"},
    title="Long-Distance Weekday-Hour Late Share Heatmap",
)
fig.update_layout(coloraxis_colorbar_tickformat=".0%")
apply_default_layout(fig, height=470)
fig.show()

**Caption - Figure 16.** Long-distance trains have very high late shares in late-night windows, but those cells need volume checks before becoming headline claims.

In [ ]:
weekday_hour_volume = weekday_hour.pivot(index="weekday_label", columns="hour_of_day", values="stop_count").reindex(WEEKDAY_ORDER)
fig = px.imshow(
    weekday_hour_volume,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Hour of day", "y": "Weekday", "color": "Stops"},
    title="Weekday-Hour Stop Volume Heatmap",
)
apply_default_layout(fig, height=470)
fig.show()

**Caption - Figure 17.** Volume is concentrated in daytime and early evening; this guards against overinterpreting high late shares in low-volume overnight cells.

## Station And Route Patterns

In [ ]:
station_type_outliers = query("""
SELECT
  station_name,
  train_type,
  SUM(stop_count) AS stop_count,
  SUM(stop_count * late_share_6_min) / SUM(stop_count) AS late_share_6_min,
  SUM(stop_count * avg_delay_min) / SUM(stop_count) AS avg_delay_min,
  AVG(p90_delay_min) AS p90_delay_min
FROM station_day
WHERE station_name IS NOT NULL
GROUP BY 1, 2
HAVING SUM(stop_count) >= 10000
ORDER BY late_share_6_min DESC, stop_count DESC
LIMIT 20
""")
station_type_outliers["station_train_type"] = station_type_outliers["station_name"] + " / " + station_type_outliers["train_type"]
fig = px.bar(
    station_type_outliers.sort_values("late_share_6_min"),
    x="late_share_6_min",
    y="station_train_type",
    orientation="h",
    color="train_type",
    text="stop_count",
    labels={"late_share_6_min": "Late share 6+ min", "station_train_type": "Station / train type"},
    title="Highest Late-Share Station And Train-Type Pairs",
)
as_percent_axis(apply_default_layout(fig, height=650), "x")
fig.show()

**Caption - Figure 18.** High-late station pairs are dominated by ICE at specific hubs, especially in western and central corridors; station effects should be analyzed within train type.

In [ ]:
station_contribution = query("""
SELECT
  station_name,
  SUM(stop_count) AS stop_count,
  SUM(stop_count * late_share_6_min) AS estimated_late_stops,
  SUM(stop_count * late_share_6_min) / SUM(stop_count) AS late_share_6_min
FROM station_day
WHERE station_name IS NOT NULL
GROUP BY 1
HAVING SUM(stop_count) >= 100000
ORDER BY estimated_late_stops DESC
LIMIT 20
""")
fig = px.bar(
    station_contribution.sort_values("estimated_late_stops"),
    x="estimated_late_stops",
    y="station_name",
    orientation="h",
    color="late_share_6_min",
    color_continuous_scale="YlOrRd",
    labels={"estimated_late_stops": "Estimated late stops", "station_name": "Station", "late_share_6_min": "Late share"},
    title="Largest Station Contributions To Late Stops",
)
fig.update_layout(coloraxis_colorbar_tickformat=".0%")
apply_default_layout(fig, height=650)
fig.show()

**Caption - Figure 19.** Contribution ranking balances volume and lateness: the most important stations for the story are not always the stations with the highest late share.

In [ ]:
line_outliers = query("""
SELECT
  train_type,
  train_name,
  final_destination_station,
  stop_count,
  late_share_6_min,
  avg_delay_min,
  p90_delay_min,
  cancellation_share
FROM lines
WHERE stop_count >= 500
ORDER BY late_share_6_min DESC, stop_count DESC
LIMIT 20
""")
line_outliers["service"] = line_outliers["train_name"] + " -> " + line_outliers["final_destination_station"].fillna("unknown")
fig = px.bar(
    line_outliers.sort_values("late_share_6_min"),
    x="late_share_6_min",
    y="service",
    orientation="h",
    color="train_type",
    labels={"late_share_6_min": "Late share 6+ min", "service": "Service"},
    title="Highest Late-Share Services With At Least 500 Stops",
)
as_percent_axis(apply_default_layout(fig, height=650), "x")
fig.show()

**Caption - Figure 20.** Route/service outliers are mostly long-distance, but the 500-stop floor is still a sensitivity choice; Phase 4 should test higher floors for headline rankings.

In [ ]:
fig = px.scatter(
    line_outliers,
    x="late_share_6_min",
    y="p90_delay_min",
    size="stop_count",
    color="train_type",
    hover_name="service",
    labels={"late_share_6_min": "Late share 6+ min", "p90_delay_min": "P90 delay (min)", "stop_count": "Stops"},
    title="Service Late Share Versus Severe Delay Tail",
)
as_percent_axis(apply_default_layout(fig), "x")
fig.show()

**Caption - Figure 21.** High late share and high p90 delay are related but not identical; the final analysis should distinguish frequent modest lateness from severe tail lateness.

## EDA Findings And Open Questions

Findings to carry into Phase 4:

1. **Coverage is the first-order issue.** The dataset is a 107-station panel through October 2025, then expands to more than 5,000 stations. Use the stable-station subset for trend claims and all stations for post-expansion operational mapping.
2. **Long-distance trains are structurally different.** ICE/IC services have much higher 6+ minute late share and p90 delay than local services. Do not combine them without stratification.
3. **Evening windows matter.** All-service late share is highest around late afternoon/evening; long-distance trains also show high late-night cells, but volume is lower there.
4. **Station rankings depend on the metric.** Highest late-share station/type pairs differ from highest contribution stations, so a final dashboard should include both views.
5. **Cancellation is not a proxy for delay.** Train types and months with higher cancellation share are not always the same ones with higher late share.

Things that were confusing and how they were resolved:

- The November/December jump looked like an operational shock at first. It is a source coverage change, resolved by comparing all stations with the stable Jan-October station panel.
- Delay values included extreme negatives and positives. The cleaning step keeps the stop event but sets delays outside [-60, 720] to NULL, preserving volume while excluding implausible delay minutes.
- Overnight long-distance late-share cells looked extreme. The volume heatmap shows those cells have fewer stops, so they are useful leads but risky headlines.
- The current data cannot fully answer "since July 2024" because only 2025 files are local. Either download 2024 months or adjust the final question to 2025.

Phase 4 recommendation: frame the analysis as descriptive decomposition. Use stable-station monthly trends for the headline trend, then decompose late-stop contribution by train type, station, hour, and route/service.